In [235]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from typing import List
from tqdm import tqdm

%matplotlib inline

## 📚 Импорт библиотек

Загружаем необходимые инструменты:
- `pandas`, `numpy` - работа с данными и числовыми операциями
- `matplotlib` - визуализация
- `tqdm` - прогресс-бары для итераций

## Считаем данные соревнования от MTS по RecSys

Датасет можно скачать отсюда: [ссылка](https://github.com/MobileTeleSystems/RecTools/tree/main/datasets/KION)

In [240]:
df = pd.read_csv('../../interactions.csv')
df.head()

,user_id,item_id,last_watch_dt,total_dur,watched_pct
0,176549,9506,2021-05-11,4250,72.0
1,699317,1659,2021-05-29,8317,100.0
2,656683,7107,2021-05-09,10,0.0
3,864613,7638,2021-07-05,14483,100.0
4,964868,9506,2021-04-30,6725,100.0


### 🔍 Анализ данных

Смотрим на размерность датасета:
- **962,179 уникальных пользователей**
- **15,706 уникальных фильмов/сериалов**
- Период данных: март - август 2021

In [241]:
df.user_id.nunique(), df.item_id.nunique()

(962179, 15706)

In [242]:
df.last_watch_dt.min(), df.last_watch_dt.max()

('2021-03-13', '2021-08-22')

### 📅 Преобразование дат

Конвертируем даты в числовой формат (дни с начала периода):
- Упрощает работу с временными рядами
- Позволяет использовать время как признак в моделях
- День 0 = 13 марта 2021

In [243]:
df['last_watch_dt'] = (pd.to_datetime(df['last_watch_dt']) - pd.to_datetime(df['last_watch_dt']).min())
df['last_watch_dt'] = df.last_watch_dt.apply(lambda x: int(str(x).split()[0]))
df.sample(5)

,user_id,item_id,last_watch_dt,total_dur,watched_pct
4724761,392623,4880,125,19959,9.0
5441733,998560,7372,146,505,100.0
5436456,815453,9728,107,11647,100.0
3588368,749270,6086,162,5305,96.0
3693017,27501,512,153,1113,19.0


In [244]:
items = pd.read_csv('../../items.csv')
items.head()

,item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,10711,film,Поговори с ней,Hable con ella,2002.0,"драмы, зарубежные, детективы, мелодрамы",Испания,NaN,16.0,NaN,Педро Альмодовар,"Адольфо Фернандес, Ана Фернандес, Дарио Гранди...",Мелодрама легендарного Педро Альмодовара «Пого...,"Поговори, ней, 2002, Испания, друзья, любовь, ..."
1,2508,film,Голые перцы,Search Party,2014.0,"зарубежные, приключения, комедии",США,NaN,16.0,NaN,Скот Армстронг,"Адам Палли, Брайан Хаски, Дж.Б. Смув, Джейсон ...",Уморительная современная комедия на популярную...,"Голые, перцы, 2014, США, друзья, свадьбы, прео..."
2,10716,film,Тактическая сила,Tactical Force,2011.0,"криминал, зарубежные, триллеры, боевики, комедии",Канада,NaN,16.0,NaN,Адам П. Калтраро,"Адриан Холмс, Даррен Шалави, Джерри Вассерман,...",Профессиональный рестлер Стив Остин («Все или ...,"Тактическая, сила, 2011, Канада, бандиты, ганг..."
3,7868,film,45 лет,45 Years,2015.0,"драмы, зарубежные, мелодрамы",Великобритания,NaN,16.0,NaN,Эндрю Хэй,"Александра Риддлстон-Барретт, Джеральдин Джейм...","Шарлотта Рэмплинг, Том Кортни, Джеральдин Джей...","45, лет, 2015, Великобритания, брак, жизнь, лю..."
4,16268,film,Все решает мгновение,NaN,1978.0,"драмы, спорт, советские, мелодрамы",СССР,NaN,12.0,Ленфильм,Виктор Садовский,"Александр Абдулов, Александр Демьяненко, Алекс...",Расчетливая чаровница из советского кинохита «...,"Все, решает, мгновение, 1978, СССР, сильные, ж..."


Получим semantic-ids через https://github.com/Kuaishou-OneRec/OpenOneRec/blob/main/tokenizer/train_res_kmeans.py

!pip3 install gensim

In [10]:
#!pip3 install gensim

In [245]:
# =========================================
# FastText embedding training from scratch
# Dataset: items (pandas DataFrame)
# =========================================

import re
import numpy as np
import pandas as pd
from gensim.models import FastText

# --- 1. Select text columns ---
text_cols = [
    "title",
    "title_orig",
    "genres",
    "countries",
    "studios",
    "directors",
    "actors",
    "description",
    "keywords"
]

# --- 2. Simple text preprocessing ---
def preprocess(text):
    if pd.isna(text):
        return []
    text = str(text).lower()
    text = re.sub(r"[^a-zа-я0-9]+", " ", text)
    tokens = text.split()
    return tokens

# --- 3. Build corpus (list of token lists) ---
corpus = []

for _, row in items.iterrows():
    tokens = []
    for col in text_cols:
        tokens.extend(preprocess(row[col]))
    if tokens:
        corpus.append(tokens)

# --- 4. Train FastText from scratch ---
fasttext_model = FastText(
    sentences=corpus,
    vector_size=128,
    window=5,
    min_count=2,
    workers=4,
    sg=1,          # skip-gram
    epochs=10
)

# --- 5. Build item-level embeddings (mean pooling) ---
def get_item_vector(row):
    tokens = []
    for col in text_cols:
        tokens.extend(preprocess(row[col]))
    vectors = [fasttext_model.wv[t] for t in tokens if t in fasttext_model.wv]
    if len(vectors) == 0:
        return np.zeros(fasttext_model.vector_size)
    return np.mean(vectors, axis=0)

item_embeddings = np.vstack(items.apply(get_item_vector, axis=1).values)

print("Embeddings shape:", item_embeddings.shape)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Embeddings shape: (15963, 128)


In [246]:
import polars as pl

# item_embeddings: numpy array (n_items, emb_dim)

result_df = pl.DataFrame({
    "pid": items["item_id"].values,
    "embedding": item_embeddings.tolist()  # one list per row
})

result_df.write_parquet("item_fasttext_embeddings.parquet")

print("Saved to item_fasttext_embeddings.parquet")
print(result_df.schema)
print(result_df.head())

Saved to item_fasttext_embeddings.parquet
Schema({'pid': Int64, 'embedding': List(Float64)})
shape: (5, 2)
┌───────┬─────────────────────────────────┐
│ pid   ┆ embedding                       │
│ ---   ┆ ---                             │
│ i64   ┆ list[f64]                       │
╞═══════╪═════════════════════════════════╡
│ 10711 ┆ [-0.04502, 0.059008, … 0.14930… │
│ 2508  ┆ [-0.081061, -0.087532, … 0.203… │
│ 10716 ┆ [-0.026725, -0.023131, … 0.202… │
│ 7868  ┆ [-0.077944, 0.035889, … 0.1840… │
│ 16268 ┆ [-0.020329, 0.148834, … 0.1083… │
└───────┴─────────────────────────────────┘


In [247]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

!KMP_DUPLICATE_LIB_OK=TRUE && python3 train_res_kmeans.py \
    --data_path item_fasttext_embeddings.parquet \
    --model_path checkpoints \
    --n_layers 3 \
    --codebook_size 64 \
    --dim 128

Total files: 1
Reading files: 100%|██████████████████████████████| 1/1 [00:00<00:00, 16.74it/s]
Final shape: (15963, 128)
data is ready
model is ready (15963, 128)
Layer 0 finished, loss={'loss': 0.003351513296365738, 'rel_loss': 0.3245384097099304}
Layer 1 finished, loss={'loss': 0.0026995218358933926, 'rel_loss': 0.30408230423927307}
Layer 2 finished, loss={'loss': 0.002365022897720337, 'rel_loss': 0.2916426658630371}
training is finished
Model saved to checkpoints/model.pt


In [248]:
!python3 infer_res_kmeans.py \
    --model_path checkpoints/model.pt \
    --emb_path item_fasttext_embeddings.parquet \
    --output_path codes.parquet

Loading model from checkpoints/model.pt
Model loaded: n_layers=3, codebook_size=64, dim=128
Loading embeddings from item_fasttext_embeddings.parquet
Embeddings shape: torch.Size([15963, 128]), num pids: 15963
Encoding embeddings...
  Processed 10000/15963
Output codes shape: torch.Size([15963, 3])
Codes saved to codes.parquet

Computing reconstruction loss...
Reconstruction loss (MSE): 0.002332
Relative loss: 0.290000


In [249]:
codes = pl.read_parquet('codes.parquet').rename({'pid':'item_id'})
codes = codes.with_columns( 
    pl.col("codes")
    .list.eval(pl.element().cast(pl.Utf8))  # <- .list.eval(), not .arr.eval()
    .list.join("-")
    .alias("codes_str")
    )
codes.head()

item_id,codes,codes_str
i64,list[i64],str
10711,"[47, 24, 2]","""47-24-2"""
2508,"[49, 27, 43]","""49-27-43"""
10716,"[33, 21, 62]","""33-21-62"""
7868,"[36, 45, 7]","""36-45-7"""
16268,"[4, 35, 25]","""4-35-25"""


In [250]:
items = pl.read_csv('../../items.csv')
items = items.join(codes, on=['item_id'], how='inner')
items.head()

item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords,codes,codes_str
i64,str,str,str,f64,str,str,f64,f64,str,str,str,str,str,list[i64],str
10711,"""film""","""Поговори с ней""","""Hable con ella""",2002.0,"""драмы, зарубежные, детективы, …","""Испания""",null,16.0,null,"""Педро Альмодовар""","""Адольфо Фернандес, Ана Фернанд…","""Мелодрама легендарного Педро А…","""Поговори, ней, 2002, Испания, …","[47, 24, 2]","""47-24-2"""
2508,"""film""","""Голые перцы""","""Search Party""",2014.0,"""зарубежные, приключения, комед…","""США""",null,16.0,null,"""Скот Армстронг""","""Адам Палли, Брайан Хаски, Дж.Б…","""Уморительная современная комед…","""Голые, перцы, 2014, США, друзь…","[49, 27, 43]","""49-27-43"""
10716,"""film""","""Тактическая сила""","""Tactical Force""",2011.0,"""криминал, зарубежные, триллеры…","""Канада""",null,16.0,null,"""Адам П. Калтраро""","""Адриан Холмс, Даррен Шалави, Д…","""Профессиональный рестлер Стив …","""Тактическая, сила, 2011, Канад…","[33, 21, 62]","""33-21-62"""
7868,"""film""","""45 лет""","""45 Years""",2015.0,"""драмы, зарубежные, мелодрамы""","""Великобритания""",null,16.0,null,"""Эндрю Хэй""","""Александра Риддлстон-Барретт, …","""Шарлотта Рэмплинг, Том Кортни,…","""45, лет, 2015, Великобритания,…","[36, 45, 7]","""36-45-7"""
16268,"""film""","""Все решает мгновение""",null,1978.0,"""драмы, спорт, советские, мелод…","""СССР""",null,12.0,"""Ленфильм""","""Виктор Садовский""","""Александр Абдулов, Александр Д…","""Расчетливая чаровница из совет…","""Все, решает, мгновение, 1978, …","[4, 35, 25]","""4-35-25"""


In [256]:
items.filter(pl.col('codes_str').str.contains(r'38-27-10'))


item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords,codes,codes_str
i64,str,str,str,f64,str,str,f64,f64,str,str,str,str,str,list[i64],str
3594,"""film""","""Риддик""","""Riddick""",2013.0,"""боевики, фантастика, триллеры,…","""Канада, США""",null,16.0,null,"""Дэвид Туи""","""Вин Дизель, Карл Урбан, Хорди …","""Преданный своими и брошенный у…","""антиутопия, жажда мести, инопл…","[38, 27, 10]","""38-27-10"""
16499,"""film""","""Экстрасенсы""","""Solace""",2015.0,"""фантастика, триллеры, криминал…","""США""",null,18.0,null,"""Афонсо Пойарт""","""Эбби Корниш, Колин Фаррелл, Хо…","""Джон Клэнси, экстрасенс, много…","""фбр, триллер, серийный убийца,…","[38, 27, 10]","""38-27-10"""
4028,"""film""","""Судная ночь 2""","""The Purge: Anarchy""",2014.0,"""боевики, триллеры, криминал""","""США, Франция""",null,18.0,null,"""Джеймс ДеМонако""","""Фрэнк Грилло, Кармен Эджого, З…","""Добро пожаловать в идеальный м…","""автобус, штурмовая винтовка, с…","[38, 27, 10]","""38-27-10"""
13167,"""film""","""Рейд 2""","""The Raid 2: Berandal""",2014.0,"""боевики""","""Индонезия, США""",null,18.0,null,"""Гарет Эванс""","""Ико Уайс, Арифин Путра, Тио Па…","""Из трех полицейских, выбравших…","""тюрьма, под прикрытием, борьба…","[38, 27, 10]","""38-27-10"""
10022,"""film""","""Остров фантазий""","""Fantasy Island""",2020.0,"""фантастика, ужасы, триллеры""","""США""",null,16.0,null,"""Джефф Уодлоу""","""Майкл Пенья, Мэгги Кью, Люси Х…","""Загадочный мистер Рорк воплоща…","""основано на тв сериале, 2020-е…","[38, 27, 10]","""38-27-10"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
14678,"""film""","""Ганмен""","""The Gunman""",2015.0,"""криминал, детективы, драмы, тр…","""США""",null,18.0,null,"""Пьер Морель""","""Хавьер Бардем, Шон Пенн, Рэй У…","""Джим Террье — ганмен, междунар…","""убийство, ударный отряд, наемн…","[38, 27, 10]","""38-27-10"""
14762,"""film""","""Мачете""","""MACHETE""",2010.0,"""боевики, триллеры, криминал, к…","""США""",null,16.0,null,"""Роберт Родригес, Этан Маникис""","""Дэнни Трехо, Джессика Альба, М…","""Во время покушения на сенатора…","""потеря любимого человека, неле…","[38, 27, 10]","""38-27-10"""
5171,"""film""","""Курьер""","""The Courier""",2019.0,"""боевики, драмы, триллеры, крим…","""Великобритания, США""",null,18.0,null,"""Закари Адлер""","""Ольга Куриленко, Гари Олдман, …","""На первый взгляд, она — всего …","""трясущаяся камера, 2010-е, Авт…","[38, 27, 10]","""38-27-10"""


In [254]:
items.group_by('codes_str').agg(pl.col('item_id').count()).sort(by='item_id', descending=False)

codes_str,item_id
str,u32
"""33-20-62""",1
"""3-39-29""",1
"""39-57-39""",1
"""47-45-0""",1
"""24-50-34""",1
…,…
"""31-13-32""",10
"""11-37-44""",12
"""14-43-14""",13


In [94]:
items.filter(pl.col('codes_str') == "12-24-18")

item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords,codes,codes_str
i64,str,str,str,f64,str,str,f64,f64,str,str,str,str,str,list[i64],str
7858,"""film""","""Концерт Дискотека Авария 04.06…","""Koncert Diskoteka Avariya 04.0…",2020.0,"""концерт, музыкальные""","""Россия""",null,0.0,null,null,"""Алексей Рыжов, Алексей Серов, …","""На МТС ТВ продолжается серия к…","""2020, россия, концерт, дискоте…","[12, 24, 18]","""12-24-18"""
2026,"""film""","""Концерт группа Секрет 03.05.20…","""Koncert gruppa Sekret 03.05.20…",2020.0,"""концерт, музыкальные""","""Россия""",null,12.0,null,null,"""Максим Леонидов, Николай Фомен…","""На МТС ТВ продолжается серия к…","""мтс, концерт, секрет, рок, liv…","[12, 24, 18]","""12-24-18"""
14872,"""film""","""Концерт Alai Oli 26.08.2020""","""Koncert Alai Oli 26.08.2020""",2020.0,"""концерт, музыкальные""","""Россия""",null,0.0,null,null,"""Ольга Маркес, Александр Шаповс…","""Погрузитесь в атмосферу петерб…","""2020, россия, концерт, alai, o…","[12, 24, 18]","""12-24-18"""
12635,"""film""","""Концерт Gruppa Skryptonite 21.…","""Koncert Gruppa Skryptonite 21.…",2020.0,"""концерт, музыкальные""","""Россия""",null,0.0,null,null,"""Скриптонит""","""На МТС ТВ продолжается серия к…","""2020, россия, концерт, gruppa,…","[12, 24, 18]","""12-24-18"""
10534,"""film""","""Концерт группа Пелагея 19.04.2…","""Koncert gruppa Pelageya 19.04.…",2020.0,"""концерт, музыкальные""","""Россия""",null,12.0,null,null,null,"""На МТС ТВ продолжается серия к…","""2020, россия, концерт, группа,…","[12, 24, 18]","""12-24-18"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
8601,"""film""","""Концерт Pizza 23.08.2020""","""Koncert Pizza 23.08.2020""",2020.0,"""концерт, музыкальные""","""Россия""",null,0.0,null,null,"""Сергей Приказчиков, Татьяна Пр…","""Прочувствуйте атмосферу концер…","""2020, россия, концерт, pizza, …","[12, 24, 18]","""12-24-18"""
13120,"""film""","""Концерт Антоха МС, Пасош, Нерв…","""Koncert Antoha MS, Pasosh, Ner…",2020.0,"""концерт, музыкальные""","""Россия""",null,12.0,null,null,"""Тима ищет свет, Антоха МС, Кис…","""На МТС ТВ продолжается серия к…","""2020, россия, концерт, антоха,…","[12, 24, 18]","""12-24-18"""
4960,"""film""","""Концерт Группа Anacondaz 10.06…","""Koncert Gruppa Anacondaz 10.06…",2020.0,"""концерт, музыкальные""","""Россия""",null,0.0,null,null,"""Сергей Карамушкин, Артём Хорев…","""На МТС ТВ продолжается серия к…","""2020, россия, концерт, группа,…","[12, 24, 18]","""12-24-18"""


Теперь надо делать обучение модели

df

## 🤖 Обучение GPT-2 для рекомендаций

В этой части мы обучим маленькую GPT-2 модель с нуля для задачи рекомендаций.

**Идея:**
- Представляем историю просмотров пользователя как последовательность semantic codes (codes_str)
- Обучаем авторегрессионную модель предсказывать следующий токен (next-token prediction)
- После обучения модель может генерировать рекомендации, продолжая историю пользователя

**Архитектура:**
- Transformer decoder (GPT-2 style)
- Маленькие параметры для быстрого прототипирования:
  - 2 слоя
  - 2 attention heads
  - 64-мерное embedding
  - Словарь: все уникальные codes_str из датасета

### 📊 Подготовка данных для GPT-2

Подготовим последовательности для обучения:
- Возьмем 100 случайных пользователей из train
- Для каждого пользователя получим их историю взаимодействий
- Преобразуем item_id в codes_str (semantic codes)

In [257]:
# Берем 100 случайных пользователей для обучения GPT-2
n_gpt_users = 5000
np.random.seed(42)
gpt_users = np.random.choice(df.user_id.unique(), size=n_gpt_users, replace=False)

# Фильтруем данные
gpt_train_df = df[df.user_id.isin(gpt_users)].copy()
gpt_train_df = gpt_train_df.sort_values(['user_id', 'last_watch_dt'])

# Создаем маппинг item_id -> codes_str
item_to_code = dict(zip(items.to_pandas()['item_id'], items.to_pandas()['codes_str']))

# Группируем по пользователям и получаем последовательности codes_str
user_sequences = []
for user_id in gpt_users:
    user_items = gpt_train_df[gpt_train_df.user_id == user_id].sort_values('last_watch_dt')['item_id'].values
    # Преобразуем в codes_str
    sequence = [item_to_code.get(item) for item in user_items if item_to_code.get(item) is not None]
    if len(sequence) >= 3:  # Берем только последовательности длиной >= 3
        user_sequences.append(sequence)

print(f"Количество пользователей: {len(user_sequences)}")
print(f"Средняя длина последовательности: {np.mean([len(s) for s in user_sequences]):.1f}")
print(f"Мин длина: {min(len(s) for s in user_sequences)}, Макс длина: {max(len(s) for s in user_sequences)}")
print(f"\nПример последовательности:\n{user_sequences[0][:10]}")

Количество пользователей: 2349
Средняя длина последовательности: 10.4
Мин длина: 3, Макс длина: 306

Пример последовательности:
['54-23-50', '20-22-41', '63-30-63']


In [258]:
user_sequences[:5]

[['54-23-50', '20-22-41', '63-30-63'],
 ['8-10-30', '35-44-23', '60-22-43', '8-39-25', '21-0-25'],
 ['20-43-0',
  '60-14-15',
  '20-22-41',
  '56-15-46',
  '20-62-11',
  '33-23-32',
  '9-29-39',
  '18-23-11',
  '38-24-21'],
 ['35-5-52',
  '20-22-41',
  '56-15-46',
  '42-3-49',
  '57-7-11',
  '33-23-32',
  '20-43-0',
  '20-9-43'],
 ['24-57-60', '6-57-25', '9-59-4']]

In [259]:
train_texts = [" ".join(seq[:-1]) for seq in user_sequences]
targets = [seq[-1] for seq in user_sequences]

In [260]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers, processors
from datasets import Dataset
import torch

# Инициализация BPE токенизатора
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(vocab_size=100, special_tokens=["<pad>", "<unk>", "<bos>", "<eos>"])

# Тренировка токенизатора
tokenizer.train_from_iterator(train_texts, trainer=trainer)

# Добавляем post-processor для GPT-2 (bos/eos)
tokenizer.post_processor = processors.TemplateProcessing(
    single="<bos> $A <eos>",
    special_tokens=[("<bos>", tokenizer.token_to_id("<bos>")), ("<eos>", tokenizer.token_to_id("<eos>"))]
)

# Тест
ids = tokenizer.encode("31-49-20 53-39-6").ids
print(ids)


[2, 55, 4, 51, 4, 17, 57, 4, 43, 4, 11, 3]




In [205]:
tokenizer

Tokenizer(version="1.0", truncation=None, padding=None, added_tokens=[{"id":0, "content":"<pad>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":1, "content":"<unk>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":2, "content":"<bos>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":3, "content":"<eos>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}], normalizer=None, pre_tokenizer=Whitespace(), post_processor=TemplateProcessing(single=[SpecialToken(id="<bos>", type_id=0), Sequence(id=A, type_id=0), SpecialToken(id="<eos>", type_id=0)], pair=[Sequence(id=A, type_id=0), Sequence(id=B, type_id=1)], special_tokens={"<bos>":SpecialToken(id="<bos>", ids=[2], tokens=["<bos>"]), "<eos>":SpecialToken(id="<eos>", ids=[3], tokens=["<eos>"])}), decoder=None, model=BPE(dropout=None, unk_token=None, continuing_su

In [261]:
class UserSeqDataset(torch.utils.data.Dataset):
    def __init__(self, texts, tokenizer, max_len=32):
        self.tokenizer = tokenizer
        self.input_ids = []
        self.labels = []
        for txt in texts:
            encoded = tokenizer.encode(txt).ids
            encoded = encoded[-max_len:]
            self.input_ids.append(torch.tensor(encoded))
            self.labels.append(torch.tensor(encoded))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {"input_ids": self.input_ids[idx], "labels": self.labels[idx]}


class CustomDataCollator:
    def __init__(self, pad_token_id=0):
        self.pad_token_id = pad_token_id

    def __call__(self, features):
        input_ids = [f["input_ids"] for f in features]
        labels = [f["labels"] for f in features]

        input_ids = pad_sequence(input_ids, batch_first=True, padding_value=self.pad_token_id)
        labels = pad_sequence(labels, batch_first=True, padding_value=-100)  # <- важно для loss

        return {"input_ids": input_ids, "labels": labels}

In [262]:
from transformers import GPT2Config, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling

config = GPT2Config(
    vocab_size=tokenizer.get_vocab_size(),
    n_positions=300,
    n_ctx=32,
    n_embd=128,
    n_layer=4,
    n_head=4,
)

model = GPT2LMHeadModel(config)



In [263]:
training_args = TrainingArguments(
    output_dir="./gpt2_userseq",
    per_device_train_batch_size=2,
    num_train_epochs=10,
    save_steps=5000,
    logging_steps=100,
    learning_rate=1e-3,
    weight_decay=0.01,
    fp16=False,
    report_to="none",
)

data_collator = CustomDataCollator(pad_token_id=tokenizer.token_to_id("<pad>"))

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator
)


trainer.train()

/Users/o.a.lashinin/miniforge3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
100,2.585850
200,2.296182
300,2.247650
400,2.162030
500,2.118096
600,2.154973
700,2.108599
800,2.082450
900,2.113419
1000,2.040821


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 61.08it/s]


TrainOutput(global_step=2350, training_loss=2.062894050111162, metrics={'train_runtime': 52.5668, 'train_samples_per_second': 89.22, 'train_steps_per_second': 44.705, 'total_flos': 1623300825600.0, 'train_loss': 2.062894050111162, 'epoch': 10.0})

In [264]:

sequence = ' '.join(user_sequences[0])
print(sequence)

input_ids = torch.tensor(tokenizer.encode(sequence).ids, device=device).unsqueeze(0)
print(input_ids)
# Маска: 1 для реальных токенов, 0 для паддинга
attention_mask = (input_ids != pad_id).long()

output = model.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,
    max_length=input_ids.shape[1]+15,
    do_sample=True,
    top_k=20,
    temperature=0.7,
    pad_token_id=pad_id,
    eos_token_id=eos_id
)
print(output[len(input_ids):])

54-23-50 20-22-41 63-30-63
tensor([[ 2, 34,  4, 23,  4, 49, 17,  4, 18,  4, 32, 16,  4, 25,  4, 16,  3]],
       device='mps:0')
tensor([], device='mps:0', size=(0, 22), dtype=torch.int64)


In [265]:

import torch.nn.functional as F
import math

device = "mps" if torch.backends.mps.is_available() else "cpu"
model.eval()

def ndcg_at_k(preds, target_ids, k=10):
    """
    preds: list of predicted token ids
    target_ids: list of true token ids
    """
    dcg = 0.0
    for i, pred in enumerate(preds[:k]):
        if pred == target_ids[0]:
            dcg = 1 / math.log2(i+2)  # i+2 т.к. логарифм с 1-based
            break
    idcg = 1.0
    return dcg / idcg

# Генерация
device = "mps" if torch.backends.mps.is_available() else "cpu"
model.to(device)
pad_id = tokenizer.token_to_id("<pad>")
eos_id = tokenizer.token_to_id("<eos>")

for seq, tgt in zip(train_texts, targets):
    input_ids = torch.tensor(tokenizer.encode(seq).ids, device=device).unsqueeze(0)
    
    # Маска: 1 для реальных токенов, 0 для паддинга
    attention_mask = (input_ids != pad_id).long()
    
    output = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_length=input_ids.shape[1]+15,
        do_sample=False,
        pad_token_id=pad_id,
        eos_token_id=eos_id
    )
    
    gen_tokens = output[0][input_ids.shape[1]:].tolist()
    gen_tokens = [x for x in gen_tokens if x not in [4]]
    real_items = []
    for i in range(0, len(gen_tokens), 3):
        try:
            real_items.append([gen_tokens[i], gen_tokens[i+1],gen_tokens[i+2]])
        except:
            print('Error in generation')
            continue


    str_items = []
    #print(real_items)
    for el in real_items:
        str_items.append('-'.join(map(str, el)))
    
    true_token_id = tokenizer.encode(tgt).ids[1:-1]
    true_token_id = '-'.join(map(str, [x for x in true_token_id if x not in [4]]))
    print(str_items, true_token_id)

    ndcg = ndcg_at_k(gen_tokens, [true_token_id], k=15)
    
    if ndcg > 0:
        print('Hoorah!', ndcg)


['14-39-3'] 16-25-16
['16-11-3'] 28-5-47
['26-37-3'] 15-24-28
['26-37-3'] 17-14-29
['15-6-3'] 14-66-9
['16-11-3'] 35-19-20
['14-39-3'] 36-12-22
['15-6-3'] 17-14-29
['15-6-3'] 19-28-20
['26-11-20', '26-11-20', '16-11-20'] 35-36-37
['26-11-3'] 44-14-47
['16-11-3'] 24-28-16
['26-11-3'] 19-31-20
['16-11-3'] 5-54-27
Error in generation
['39-11-11'] 16-66-12
['26-37-3'] 40-43-10
['15-6-3'] 53-20-41
['16-11-3'] 16-29-43
['26-37-3'] 51-65-58
['26-11-3'] 17-18-32
['16-11-3'] 34-42-27
['15-6-3'] 41-54-22
['16-6-20', '26-11-3'] 61-26-47
['16-11-3'] 39-14-14
['26-39-3'] 45-18-7
['16-11-3'] 30-23-48
['26-11-3'] 44-14-53
['26-40-3'] 36-12-22
['16-11-3'] 52-29-45
['16-11-3'] 63-27-22
['15-6-3'] 56-16-40
['26-37-3'] 36-12-22
['14-39-3'] 44-21-23
['15-6-3'] 36-12-22
['26-11-3'] 44-19-17
['16-11-3'] 42-41-21
['16-28-3'] 19-36-33
['26-31-17', '26-37-3'] 44-28-45
['39-11-26', '26-37-3'] 35-19-20
['15-6-3'] 40-26-21
['16-11-3'] 51-65-7
['16-11-3'] 46-44-10
['26-40-3'] 20-14-60
['16-11-3'] 17-18-32
['16-11-

KeyboardInterrupt: 

In [266]:
from collections import Counter 

popular = [t[0] for t in Counter(' '.join(train_texts).split()).most_common()][:10]
popular = ' '.join(popular)
popular

'20-22-41 56-15-46 57-7-11 33-23-32 51-22-2 3-39-21 60-14-15 60-43-41 20-62-11 20-43-0'

In [267]:
target_sequence = "20-22-41 56-15-46 57-7-11 33-23-32 51-22-2 3-39-21 60-14-15 60-43-41 20-62-11 20-43-0"
target_ids = tokenizer.encode(target_sequence).ids

In [268]:
class SupervisedDataset(torch.utils.data.Dataset):
    def __init__(self, tokenizer, target_ids, n_samples=100):
        self.input_ids = []
        self.labels = []

        for _ in range(n_samples):
            # input: [BOS] + target[:-1]
            inp = [tokenizer.token_to_id("<bos>")] + target_ids[:-1]
            lbl = target_ids  # полная последовательность как labels
            self.input_ids.append(torch.tensor(inp))
            self.labels.append(torch.tensor(lbl))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {"input_ids": self.input_ids[idx], "labels": self.labels[idx]}

In [269]:
class SFTDataCollator:
    def __init__(self, pad_token_id=0):
        self.pad_token_id = pad_token_id

    def __call__(self, features):
        input_ids = [f["input_ids"] for f in features]
        labels = [f["labels"] for f in features]

        input_ids = pad_sequence(input_ids, batch_first=True, padding_value=self.pad_token_id)
        labels = pad_sequence(labels, batch_first=True, padding_value=-100)

        return {"input_ids": input_ids, "labels": labels}

In [270]:
from transformers import Trainer, TrainingArguments

# Dataset и collator
sft_dataset = SupervisedDataset(tokenizer, target_ids, n_samples=200)
collator = SFTDataCollator(pad_token_id=tokenizer.token_to_id("<pad>"))

# Аргументы тренировки
training_args = TrainingArguments(
    output_dir="./gpt2_sft",
    per_device_train_batch_size=4,
    num_train_epochs=20,
    learning_rate=5e-4,
    weight_decay=0.01,
    logging_steps=5,
    save_steps=20,
    report_to="none",
    fp16=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=sft_dataset,
    data_collator=collator
)

trainer.train()

/Users/o.a.lashinin/miniforge3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
5,4.774261
10,3.465546
15,2.700606
20,2.317845
25,1.956387
30,1.694517
35,1.463183
40,1.297039
45,1.125334
50,0.995090


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 61.01it/s]
/Users/o.a.lashinin/miniforge3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 75.49it/s]
/Users/o.a.lashinin/miniforge3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 70.53it/s]
/Users/o.a.lashinin/miniforge3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 20.61it/s]
/Users/o.a.

TrainOutput(global_step=1000, training_loss=0.15093963790126144, metrics={'train_runtime': 19.4231, 'train_samples_per_second': 205.94, 'train_steps_per_second': 51.485, 'total_flos': 990093312000.0, 'train_loss': 0.15093963790126144, 'epoch': 20.0})

In [271]:

import torch.nn.functional as F
import math

device = "mps" if torch.backends.mps.is_available() else "cpu"
model.eval()

def ndcg_at_k(preds, target_ids, k=10):
    """
    preds: list of predicted token ids
    target_ids: list of true token ids
    """
    dcg = 0.0
    for i, pred in enumerate(preds[:k]):
        if pred == target_ids[0]:
            dcg = 1 / math.log2(i+2)  # i+2 т.к. логарифм с 1-based
            break
    idcg = 1.0
    return dcg / idcg

# Генерация
device = "mps" if torch.backends.mps.is_available() else "cpu"
model.to(device)
pad_id = tokenizer.token_to_id("<pad>")
eos_id = tokenizer.token_to_id("<eos>")

for seq, tgt in zip(train_texts, targets):
    input_ids = torch.tensor(tokenizer.encode(seq).ids, device=device).unsqueeze(0)
    
    # Маска: 1 для реальных токенов, 0 для паддинга
    attention_mask = (input_ids != pad_id).long()
    
    output = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_length=input_ids.shape[1]+100,
        do_sample=True,
        temperature=0.9,
        top_k=10,
        pad_token_id=pad_id,
        eos_token_id=eos_id
    )
    
    gen_tokens = output[0][input_ids.shape[1]:].tolist()
    gen_tokens = [x for x in gen_tokens if x not in [4]]
    real_items = []
    for i in range(0, len(gen_tokens), 3):
        try:
            real_items.append([gen_tokens[i], gen_tokens[i+1],gen_tokens[i+2]])
        except:
            print('Error in generation')
            continue


    str_items = []
    #print(real_items)
    for el in real_items:
        str_items.append('-'.join(map(str, el)))
    
    true_token_id = tokenizer.encode(tgt).ids[1:-1]
    true_token_id = '-'.join(map(str, [x for x in true_token_id if x not in [4]]))
    print(str_items, true_token_id)

    ndcg = ndcg_at_k(gen_tokens, [true_token_id], k=15)
    
    if ndcg > 0:
        print('Hoorah!', ndcg)


['22-48-45', '48-50-7', '28-50-27', '50-27-17', '39-22-17', '22-17-3'] 16-25-16
Error in generation
['43-22-17', '27-28-28', '48-39-39', '26-39-22', '17-22-17'] 28-5-47
Error in generation
['48-39-5', '28-39-30', '30-28-8', '45-30-39', '39-39-22', '39-30-48', '22-29-8', '22-22-17', '39-22-30', '5-29-22', '17-22-17', '22-17-39'] 15-24-28
['50-22-48', '22-48-39', '48-48-39', '48-48-39', '22-48-39', '39-39-32', '29-39-32', '39-32-29', '39-7-22', '39-22-22', '32-50-22', '17-8-18', '32-17-39', '39-22-22', '17-39-3'] 17-14-29
['22-48-27', '48-50-7', '28-50-27', '50-27-17', '39-22-32', '22-17-3'] 14-66-9
['18-22-29', '43-28-28', '48-39-50', '5-39-22', '17-39-22', '48-39-48', '5-28-29', '30-5-28', '8-18-32', '17-39-39', '29-32-17', '32-17-39', '50-32-17', '39-22-22', '48-39-22', '30-32-17', '22-17-39', '22-22-17', '39-22-22', '17-39-3'] 35-19-20
['40-45-48', '50-22-3'] 36-12-22
Error in generation
['29-45-48', '50-7-28', '50-27-8', '50-27-17', '39-22-17', '22-32-39', '32-32-39', '32-17-39', '2

KeyboardInterrupt: 